# <center>**Enhancing LEM-X Imaging with the IROS Reconstruction Pipeline**<center>

## <center>**Sky Reconstruction Efficiency**<center>

In [1]:
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd

from bloodmoon.mask import CodedMaskCamera, codedmask
from bloodmoon.io import simulation_files
from bloodmoon.types import CoordEquatorial
import darksun as ds
from darksun.data import Log, DataLoader, CatalogueLoader

from IROSrec.handle import config_dirpaths
import imgmaker as mgm
from imgmaker.fns import CameraUnitMap

In [2]:
# MASK_FITS: str = "mask_NTHT_20260129_CORRECTED.fits"
MASK_FITS: str = "mask_NTHT_20250725.fits"

# SKYFIELD: str = "IROSDummy"
SKYFIELD: str = "GalacticCentre"
# DATA_FITS: str = "baseline_2-50keV_1ks"
DATA_FITS: str = "galctr_rxte-sax_mask_050_1040x17_2-50keV_1ks_mask25"

RUN_ID: str = 'GC_rec_1ks_detected_2-6keV_mask25'

ID_CAMERA_A: str = "cam1a"
ID_CAMERA_B: str = "cam1b"
DATASET: str = "detected"

E_min: float = 2.0  # [keV]
E_max: float = 6.0  # [keV]
coords2exclude: list[CoordEquatorial] | None = None

UP_X, UP_Y = 2, 1
hide_bulk_els_y: float = 1.5   # [mm]

In [3]:
MASK_PATH, SIMUL_DATA_PATH, SAVE_PATH = config_dirpaths(
    mask=MASK_FITS,
    skyfield=SKYFIELD,
    simul=DATA_FITS,
    runID=RUN_ID,
)
OUT_RESULTS_PATH = mgm.config_savedata_to()

wfm: CodedMaskCamera = codedmask(MASK_PATH, UP_X, UP_Y, hide_bulk_els_y=hide_bulk_els_y)

filepaths: dict[str, dict[str, Path]] = simulation_files(SIMUL_DATA_PATH)
sdlA = ds.get_data(filepaths[ID_CAMERA_A][DATASET], E_min=E_min, E_max=E_max, coords=coords2exclude)
catA = ds.get_catalogue(filepaths[ID_CAMERA_A]['sources'])
sdlB = ds.get_data(filepaths[ID_CAMERA_B][DATASET], E_min=E_min, E_max=E_max, coords=coords2exclude)
catB = ds.get_catalogue(filepaths[ID_CAMERA_B]['sources'])

logA, logB = ds.load_database(f"{SAVE_PATH}/IROS_sources_db.fits")

# Loading data...
# Loading completed!


### <center>**Benchmark Tables**<center>

In [4]:
import re

def adjust_Tabfrmt(txt: str) -> str:
    # insert \hline instead of rules (journal guidelines)
    for rule in ('toprule', 'midrule', 'bottomrule'):
        txt = txt.replace(rule, 'hline')
    # shift caption and label at the end (journal guidelines)
    pattern = r"(\\begin\{table\}.*?)(\\caption\{.*?\})\s*(\\label\{.*?\})\s*(\\begin\{tabular\}.*?\\end\{tabular\})"
    replacement = r"\1\4\n\2\n\3"
    txt = re.sub(pattern, replacement, txt, flags=re.DOTALL)
    # convert to onecolumn
    txt = txt.replace('table', 'table*')
    return txt

def sort_by(df: pd.DataFrame, key: str, **kwargs: Any) -> pd.DataFrame:
    """Sort DataFrame wrt input column key."""
    return df.sort_values(by=[key], ascending=False, ignore_index=True, **kwargs)

In [5]:
from numpy.typing import NDArray

def comp_src_mstd(log: Log, varmap: NDArray, boxsize: tuple[int, int]) -> NDArray:
    """Computes the RMSE for each IROS source from given varmap in specified array box."""
    mstds: list[float] = []
    boxsize_ = (max(boxsize[0], 1), max(boxsize[1], 1))
    for y, x in zip(log.log['y'], log.log['x']):
        srows, scols = (
            slice(y - boxsize_[0], y + boxsize_[0] + 1),
            slice(x - boxsize_[1], x + boxsize_[1] + 1),
        )
        mstd = np.sqrt(np.mean(varmap[srows, scols]))
        mstds.append(mstd)
    return np.array(mstds)

def gather_cam_data(
    log: Log,
    catalogue: CatalogueLoader,
    sdl: DataLoader,
    camera: CodedMaskCamera,
    varmap: NDArray,
) -> pd.DataFrame:
    """
    Gathers single camera data from IROS reconstruction database.
    """
    ids = np.array([src.upper() for src in log.log['ID']])
    theta_res_x, theta_res_y = mgm.get_angularcoords_residues(log, catalogue, sdl, camera)
    cts = np.array(log.log['fluence']).round(decimals=0)
    true_cts = mgm.extract_catalogue_fluences(log, catalogue, sdl, camera)
    src_mstd = comp_src_mstd(log, varmap, boxsize=tuple(int(np.ceil(a // 2)) for a in ds.psf_extension(camera)[::-1]))
    dmap = {
        'Source': ids,
        'DthetaX': theta_res_x,
        'DthetaY': theta_res_y,
        'True_cts': true_cts,
        'IROS_cts': cts,
        'Dcts': (cts - true_cts) / src_mstd,
        'SNR': np.array(log.log['snr']),
    }
    return pd.DataFrame(dmap)

def get_joint_tab(
    data_camA: pd.DataFrame,
    data_camB: pd.DataFrame,
    unitmap: CameraUnitMap,
) -> pd.DataFrame:
    """Generates a Dataframe with output data from both cameras."""
    def _extract(data: pd.DataFrame, idxs: NDArray, camID: str) -> dict[str, NDArray]:
        cam_dmap = {
            f'DthetaX_{camID}': np.array(data['DthetaX'])[idxs],
            f'DthetaY_{camID}': np.array(data['DthetaY'])[idxs],
            f'True_cts_{camID}': np.array(data['True_cts'])[idxs],
            f'IROS_cts_{camID}': np.array(data['IROS_cts'])[idxs],
            f'Dcts_{camID}': np.array(data['Dcts'])[idxs],
        }
        return cam_dmap

    compose: Callable = lambda a, b: np.sqrt(a ** 2 + b ** 2)
    dmap = {
        'Source': np.array(data_camA['Source'])[unitmap.idx_a],
        **_extract(data_camA, unitmap.idx_a, 'A'),
        **_extract(data_camB, unitmap.idx_b, 'B'),
        'SNR': compose(
            np.array(data_camA['SNR'])[unitmap.idx_a],
            np.array(data_camB['SNR'])[unitmap.idx_b],
        ),
    }
    return pd.DataFrame(dmap)

In [6]:
from bloodmoon.mask import count, variance

def get_varmap(camera: CodedMaskCamera, sdl: DataLoader) -> NDArray:
    detector = count(camera, sdl.DLdata)[0]
    varmap = variance(camera, detector)
    return varmap


varmapA, varmapB = map(lambda sdl: get_varmap(wfm, sdl), (sdlA, sdlB))

UserInfo: using bulk mask of [0.0 x 1.5] mm.


In [7]:
ds.pixels_angular_resolution(wfm)
cu_map = mgm.get_srcmap_for_unit(logA.log['ID'], logB.log['ID'])

# Table - CAMERA A
data_camA = gather_cam_data(logA, catA, sdlA, wfm, varmapA)

# Table - CAMERA B
data_camB = gather_cam_data(logB, catB, sdlB, wfm, varmapB)


Pixel angular resolution at upscaling (x, y): (2, 1)
  - fine direction: 2.1163 arcmin
  - coarse direction: 8.4653 arcmin



Analysing LEMX-CAM1BS1: 100%|██████████| 21/21 [00:01<00:00, 15.14it/s]  


In [8]:
unit_data = get_joint_tab(data_camA, data_camB, cu_map)

KWS = {
    'label': 'Table1',
    'caption': 'Testing $`to\\_latex`$ fn.',
    'float_format': "%.4f",
    'column_format': 'l' + 'c' * (len(unit_data.columns) - 2) + 'r',
}
tab = mgm.df2TeXtab(
    df=sort_by(unit_data, 'SNR'),
    adjust_tabfrmt=adjust_Tabfrmt,
    # save_to=f'{OUT_RESULTS_PATH}/../texTable_Unit_results_{DATASET}_{E_min}-{E_max}keV.tex',
    overwrite=True,
    **KWS,
)

In [9]:
unit_data.sort_values('SNR', ascending=False, ignore_index=True)

,Source,DthetaX_A,DthetaY_A,True_cts_A,IROS_cts_A,Dcts_A,DthetaX_B,DthetaY_B,True_cts_B,IROS_cts_B,Dcts_B,SNR
0,SCOX1,0.033157,-0.238792,782305.0,779468.0,-2.707564,0.061384,-0.241896,696826.0,694072.0,-2.771863,1009.583482
1,GX5-1,0.000063,-0.359214,87287.0,86801.0,-0.426482,0.002631,1.102177,88438.0,89765.0,1.193525,109.288757
2,GX349+2,0.048058,-0.243569,57654.0,56681.0,-0.853459,0.001924,4.425942,57499.0,55919.0,-1.420713,59.955723
3,GX9+1,-0.072516,2.860509,45955.0,48633.0,2.348901,0.115127,1.262934,46741.0,48277.0,1.381428,56.597247
4,GX17+2,0.141454,1.744140,44734.0,45546.0,0.719123,-0.121056,3.874322,47708.0,45581.0,-1.915990,46.654653
5,GX13+1,0.150493,9.607190,27029.0,26604.0,-0.372777,0.079284,1.399409,26812.0,27227.0,0.373154,33.082940
6,GX3+1,0.077527,-4.413309,26883.0,28530.0,1.444952,0.031081,-8.562299,28060.0,27407.0,-0.587236,30.528719
7,GX340+0,0.037605,4.559263,24080.0,23227.0,-0.788800,-0.002161,2.202856,25446.0,25351.0,-0.088151,30.233650
8,X1820-303,-0.177949,-1.734631,20676.0,24096.0,2.999920,0.020281,-9.802002,20321.0,20923.0,0.541318,25.737277
9,GX9+9,-0.153553,0.048773,18451.0,19684.0,1.081463,0.109791,-2.511380,18791.0,15749.0,-2.735472,20.283325


In [47]:
df = pd.merge(data_camA.dropna(), data_camB.dropna(), on='Source', how='outer', suffixes=('_A', '_B'))

a = df['SNR_A'] ** 2
b = df['SNR_B'] ** 2
df['SNR'] = np.sqrt(a.add(b, fill_value=0.0))

sort_by(df.drop(columns=['SNR_A', 'SNR_B']), 'SNR')

,Source,DthetaX_A,DthetaY_A,True_cts_A,IROS_cts_A,Dcts_A,DthetaX_B,DthetaY_B,True_cts_B,IROS_cts_B,Dcts_B,SNR
0,SCOX1,0.033157,-0.238792,782305.0,779468.0,-2.707564,0.061384,-0.241896,696826.0,694072.0,-2.771863,1009.583482
1,GX5-1,0.000063,-0.359214,87287.0,86801.0,-0.426482,0.002631,1.102177,88438.0,89765.0,1.193525,109.288757
2,GX349+2,0.048058,-0.243569,57654.0,56681.0,-0.853459,0.001924,4.425942,57499.0,55919.0,-1.420713,59.955723
3,GX9+1,-0.072516,2.860509,45955.0,48633.0,2.348901,0.115127,1.262934,46741.0,48277.0,1.381428,56.597247
4,GX17+2,0.141454,1.744140,44734.0,45546.0,0.719123,-0.121056,3.874322,47708.0,45581.0,-1.915990,46.654653
5,GX13+1,0.150493,9.607190,27029.0,26604.0,-0.372777,0.079284,1.399409,26812.0,27227.0,0.373154,33.082940
6,GX3+1,0.077527,-4.413309,26883.0,28530.0,1.444952,0.031081,-8.562299,28060.0,27407.0,-0.587236,30.528719
7,GX340+0,0.037605,4.559263,24080.0,23227.0,-0.788800,-0.002161,2.202856,25446.0,25351.0,-0.088151,30.233650
8,X1820-303,-0.177949,-1.734631,20676.0,24096.0,2.999920,0.020281,-9.802002,20321.0,20923.0,0.541318,25.737277
9,GX9+9,-0.153553,0.048773,18451.0,19684.0,1.081463,0.109791,-2.511380,18791.0,15749.0,-2.735472,20.283325


In [ ]:
# def extract_unique_idxs(log: Log, unique_srcs: NDArray) -> NDArray:
#     """Extracts the source ID indexes relative to the unique sources in camera Unit."""
#     # NOTE: this logic assumes that `log.log['ID']` has unique elements;
#     #       repeating IDs will be overwritten by dict comprehension
#     dmap = {src: idx for idx, src in enumerate(log.log['ID'])}
#     idxs = sorted([dmap[src] for src in unique_srcs if src in dmap])
#     return np.array(idxs, dtype=np.int64)

def extract_unique_idxs(data: pd.DataFrame, common_idxs: NDArray) -> NDArray:
    """Extracts the source ID indexes relative to the unique sources in camera Unit."""
    return np.setdiff1d(np.arange(len(data)), common_idxs)

def gather_unique_cam_data(data: pd.DataFrame, idxs: NDArray) -> pd.DataFrame:
    return data.iloc[idxs].dropna()


idxsA, idxsB = map(lambda log: extract_unique_idxs(log, cu_map.unique), (logA, logB))
unique_dataA = gather_unique_cam_data(data_camA, idxsA)
unique_dataB = gather_unique_cam_data(data_camB, idxsB)

In [12]:
np.setdiff1d(np.arange(len(data_camA)), cu_map.idx_a)

gather_unique_cam_data(data_camB, np.setdiff1d(np.arange(len(data_camB)), cu_map.idx_b))

,Source,DthetaX,DthetaY,True_cts,IROS_cts,Dcts,SNR
13,X1636-536,-0.004210,11.948710,5648.0,8341.0,2.928724,7.916605
14,IGRJ17091-3624,0.047349,-0.386387,4939.0,7071.0,1.917380,5.363527
16,GX339-4,0.101933,-11.892784,5386.0,7586.0,2.120574,5.801377
17,X1630-472,0.257617,8.814137,4997.0,5822.0,0.793532,5.619141
18,SERX1,-0.473320,6.714591,5428.0,5173.0,-0.336090,5.847603
19,GRS1915+105,-0.094853,4.750983,3087.0,3682.0,1.481024,9.008246


In [ ]:
def gather_cam_data(
    log: Log,
    catalogue: CatalogueLoader,
    sdl: DataLoader,
    camera: CodedMaskCamera,
    varmap: NDArray,
) -> pd.DataFrame:
    """
    Gathers single camera data from IROS reconstruction database.
    """
    ids = np.array([src.upper() for src in log.log['ID']])
    theta_res_x, theta_res_y = mgm.get_angularcoords_residues(log, catalogue, sdl, camera)
    cts = np.array(log.log['fluence']).round(decimals=0)
    true_cts = mgm.extract_catalogue_fluences(log, catalogue, sdl, camera)
    src_mstd = comp_src_mstd(log, varmap, boxsize=tuple(int(np.ceil(a // 2)) for a in ds.psf_extension(camera)[::-1]))
    dmap = {
        'Source': ids,
        'DthetaX': theta_res_x,
        'DthetaY': theta_res_y,
        'True_cts': true_cts,
        'IROS_cts': cts,
        'Dcts': (cts - true_cts) / src_mstd,
        'SNR': np.array(log.log['snr']),
    }
    return pd.DataFrame(dmap)

def _extract_common_db(
    data_camA: pd.DataFrame,
    data_camB: pd.DataFrame,
    unitmap: CameraUnitMap,
) -> pd.DataFrame:
    """Generates a Dataframe with output data from both cameras."""
    def _extract(data: pd.DataFrame, idxs: NDArray, camID: str) -> dict[str, NDArray]:
        cam_dmap = {
            f'DthetaX_{camID}': np.array(data['DthetaX'])[idxs],
            f'DthetaY_{camID}': np.array(data['DthetaY'])[idxs],
            f'True_cts_{camID}': np.array(data['True_cts'])[idxs],
            f'IROS_cts_{camID}': np.array(data['IROS_cts'])[idxs],
            f'Dcts_{camID}': np.array(data['Dcts'])[idxs],
        }
        return cam_dmap

    compose: Callable = lambda a, b: np.sqrt(a ** 2 + b ** 2)
    dmap = {
        'Source': np.array(data_camA['Source'])[unitmap.idx_a],
        **_extract(data_camA, unitmap.idx_a, 'A'),
        **_extract(data_camB, unitmap.idx_b, 'B'),
        'SNR': compose(
            np.array(data_camA['SNR'])[unitmap.idx_a],
            np.array(data_camB['SNR'])[unitmap.idx_b],
        ),
    }
    return pd.DataFrame(dmap)

def get_unit_tab(
    data_camA: pd.DataFrame,
    data_camB: pd.DataFrame,
    unitmap: CameraUnitMap,
) -> pd.DataFrame:
    """
    Generates a Dataframe with output data from both cameras.
    """
    def _config_data(data: pd.DataFrame, idxs: NDArray, camID: str) -> dict[str, NDArray]:
        cam_dmap = {
            f'{col}_{camID}': 
        }

In [14]:
# def comp_fc_mstd(varmap: NDArray, boxsize: tuple[int, int]) -> float:
#     """Computes the RMSE of the given varmap in specified array box."""
#     n, m = varmap.shape
#     srows, scols = (
#         slice((n - 1) // 2 - boxsize[0], (n - 1) // 2 + boxsize[0] + 1),
#         slice((m - 1) // 2 - boxsize[1], (m - 1) // 2 + boxsize[1] + 1),
#     )
#     mstd = np.sqrt(np.mean(varmap[srows, scols]))
#     return mstd

# def gather_cam_data(
#     log: Log,
#     catalogue: CatalogueLoader,
#     sdl: DataLoader,
#     camera: CodedMaskCamera,
#     varmap: NDArray,
# ) -> pd.DataFrame:
#     """
#     Gathers single camera data from IROS reconstruction database.
#     """
#     ids = np.array([src.upper() for src in log.log['ID']])
#     theta_res_x, theta_res_y = mgm.get_angularcoords_residues(log, catalogue, sdl, camera)
#     cts = np.array(log.log['fluence'])
#     true_cts = mgm.extract_catalogue_fluences(log, catalogue, sdl, camera)
#     # rmse = comp_fc_mstd(varmap, boxsize=(camera.upscale_f.y * 80, camera.upscale_f.x * 200))
#     src_mstd = comp_src_mstd(log, varmap, boxsize=tuple(int(np.ceil(a // 2)) for a in ds.psf_extension(camera)[::-1]))
#     dmap = {
#         log.name: {
#             'Source': ids,
#             'DthetaX': theta_res_x,
#             'DthetaY': theta_res_y,
#             'IROS_cts': cts,
#             'True_cts': true_cts,
#             # 'Dcts': (cts - true_cts) / np.sqrt(true_cts),
#             'Dcts': (cts - true_cts) / src_mstd,
#             # 'Dcts_var': (cts - true_cts) / rmse,
#             'SNR': np.array(log.log['snr']),
#             # 'thetaX [deg]': np.array(log.log['angle_x']),
#             # 'thetaY [deg]': np.array(log.log['angle_y']),
#         }
#     }
#     return pd.DataFrame(dmap)

# def get_joint_tab(
#     data_camA: pd.DataFrame,
#     data_camB: pd.DataFrame,
#     unitmap: CameraUnitMap,
# ) -> pd.DataFrame:
#     """Generates a Dataframe with output data from both cameras."""
#     compose: Callable = lambda a, b: np.sqrt(a ** 2 + b ** 2)
#     dmap = {
#         'Source': np.array(data_camA.CAM1A['Source'])[unitmap.idx_a],

#         'DthetaX_A': np.array(data_camA.CAM1A['DthetaX'])[unitmap.idx_a],
#         'DthetaY_A': np.array(data_camA.CAM1A['DthetaY'])[unitmap.idx_a],
#         'TrueCts_A': np.array(data_camA.CAM1A['True_cts'])[unitmap.idx_a],
#         'ReconstrCts_A': np.array(data_camA.CAM1A['IROS_cts'])[unitmap.idx_a],
#         'Dcts_A': np.array(data_camA.CAM1A['Dcts'])[unitmap.idx_a],
#         # 'Dcts_A_var': np.array(data_camA.CAM1A['Dcts_var'])[unitmap.idx_a],

#         'DthetaX_B': np.array(data_camB.CAM1B['DthetaX'])[unitmap.idx_b],
#         'DthetaY_B': np.array(data_camB.CAM1B['DthetaY'])[unitmap.idx_b],
#         'TrueCts_B': np.array(data_camB.CAM1B['True_cts'])[unitmap.idx_b],
#         'ReconstrCts_B': np.array(data_camB.CAM1B['IROS_cts'])[unitmap.idx_b],
#         'Dcts_B': np.array(data_camB.CAM1B['Dcts'])[unitmap.idx_b],
#         # 'Dcts_B_var': np.array(data_camB.CAM1B['Dcts_var'])[unitmap.idx_b],

#         'SNR': compose(
#             np.array(data_camA.CAM1A['SNR'])[unitmap.idx_a],
#             np.array(data_camB.CAM1B['SNR'])[unitmap.idx_b],
#         ),
#     }
#     return pd.DataFrame(dmap)